[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasonCiemielewski/SVEF-Drug-Rescue/blob/main/notebooks/Midterm_Presentation.ipynb)

# Midterm Presentation

BIFX 546 Machine Learning for Bioinformatics - Spring 2026

Student: Jason Ciemielewski
Professor: Dr. Sarangan Ravichandran


## Introduction

There are many interventional drug clinical trials that reach phase 2/3 and do not succesfully complete.  These trials may not successfully complete for a variety of reasons. Trials may not successfully complete due to multiple issues such as toxicity, futility, business decisions, or administration issues.  Drug candidates in phase 2/3 trials that do not successfully complete their trial due to non toxicty reasons are particularly interesting candidates for drug repurposing.  These candidates have shown intial success in clearing toxicity hurdles and potentially represent less expensive drug development candidates.

This project aims to create the Safety Validated Efficacy Failed (SVEF) dataset to identify potential drug candidates for repurposing.  After candidates are identified, Interpretable cross-attention network (ICAN) will be used to identify potential new targets for these candidates.

### Data Background
The raw data used for this SVEF dataset comes from Aggregate Analysis of ClinicalTrials.gov (AACT) Database. (https://aact.ctti-clinicaltrials.org/downloads; Flat Text File snapshot.)  The download date of the snapshot has, embarrassingly, been lost.  A more recent snapshot flat text file will be downloaded and processed with the new iteration of the production pipeline script.  The changes in data is expected to be minimal, as the new replacement snapshot data will only contain ~ 2 months of new data over a multi decade timespan.  The zipped flat text file is >2gb, <3gb, but unzips to about 16gb of >40 .txt files.  The SVEF currently only uses 9 of those .txt files.  Instructions for running the production pipeline can be found in the README for the git hub repo at https://github.com/JasonCiemielewski/SVEF-Drug-Rescue.git.

### Clinical Trials Background
The Trial overall_status by Phase plot shows a further filtering of the data (from Phase Distribution: Interventional Drug Trials) to only include trials that have the Terminated, Suspended, Withdrawn, and Unknown status (studies.txt['overall_status]) in Phase 2, Phase 2/3, Phase 3 (studies.txt['phase']).  According to the FDA (https://www.fda.gov/patients/drug-development-process/step-3-clinical-research), Phase 1 purpose is Safety and dosage (dose finding), Phase 2 purpose is efficacy and side effects, Phase 3 is efficacy and monitoring of adverse reactions, and Phase 2/3 is

Investigational New Drug (IND) applications are required for use of a human drug product nore previously authorized for marketing in the United States. The phase of the investigation, extent of the human study, duration of the investigation, nature and source of the drug substance, and the drug product dosage form affect the type of information required for submission. (https://www.fda.gov/media/70822/download) 

## Set Up

In [ ]:
import os
import sys

# 1. IDENTIFY ENVIRONMENT & ROOT
if os.path.exists('../src'):
    # Local IDE: Running from /notebooks folder
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print("Local Environment (Notebook Folder) detected. Skipping clone.")
elif os.path.exists('src'):
    # Local IDE: Running from Root
    PROJECT_ROOT = os.getcwd()
    print("Local Environment (Root Folder) detected. Skipping clone.")
else:
    # Google Colab: Fetch the repo folders
    print("Colab Environment detected. Fetching repository...")
    REPO_URL = "https://github.com/JasonCiemielewski/SVEF-Drug-Rescue.git"
    REPO_DIR = "SVEF-Drug-Rescue"
    !git clone {REPO_URL}
    %cd {REPO_DIR}
    PROJECT_ROOT = os.getcwd()

# 2. DEFINE PATHS & ADD TO SYS.PATH
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'demo') 
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"System Ready. PROJECT_ROOT: {PROJECT_ROOT}")

### Plot Phase Distribution: Interventional Drug Trials

In [ ]:
from IPython.display import Image, display
import os

# Use the verified path from your Master Setup
img_path = os.path.join(FIGURES_DIR, 'phase_dist_drug_interven.png')

if os.path.exists(img_path):
    display(Image(filename=img_path, width=800))
else:
    print(f"⚠️ Image not found at: {img_path}")
    print("Check if 'phase_dist_drug_interven.png' exists in your /reports/figures/ folder.")

The Phase Distribution: Interventional Drug Trials plot shows the number of clinical trials recorded in the AACT database for Drug (interventions.txt['intervention_type']) and Interventional (studies.txt['study_type']) trials for each phase (studies.txt['phase']).

In [ ]:
%pip install highlight_text

### Plot Trial overall_status by Phase

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from highlight_text import fig_text

# --- 1. DIRECT DATA LOAD ---
# Points directly to your production audit file
prod_audit_file = os.path.join(PROJECT_ROOT, 'data', 'interim', 'audit', 'svef_logic_audit.csv')
df = pd.read_csv(prod_audit_file)

# --- 2. FAST FILTER & PIVOT ---
phases = ['PHASE2', 'PHASE2/PHASE3', 'PHASE3']
statuses = ['TERMINATED', 'SUSPENDED', 'WITHDRAWN', 'UNKNOWN']

# Filter and Pivot in one go
pivot_df = (df[df['phase'].isin(phases)]
            .groupby(['phase', 'overall_status']).size()
            .unstack(fill_value=0)
            .reindex(index=phases, columns=statuses, fill_value=0))

# --- 3. PLOT EXECUTION ---
totals = pivot_df.sum(axis=1)
fig, ax = plt.subplots(figsize=(11, 6), dpi=100)
x = np.arange(len(pivot_df.index))
bar_width = 0.18
colors = ["#E85252", "#E0AC2B", "#6689C6", "#555555"] # Term, Susp, With, Unk

# Background Totals
ax.bar(x, totals, width=bar_width * 5.5, color="#F2F2F2", zorder=0)

# Bars
for i, status in enumerate(pivot_df.columns):
    ax.bar(x + (i - 1.5) * bar_width, pivot_df[status], 
           width=bar_width, color=colors[i], zorder=1)

# Formatting
ax.set_xticks(x)
ax.set_xticklabels(['Phase 2', 'Phase 2/3', 'Phase 3'], fontweight='bold')
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.set_ylabel("Number of Trials")
ax.set_xlabel("Phase")

# Header & Legend
fig.text(x=0.12, y=0.96, s="Trial overall_status by Phase", size=16, weight="bold")
fig_text(x=0.12, y=0.91, 
         s="Tracking <TERMINATED>, <SUSPENDED>, <WITHDRAWN>, and <UNKNOWN>",
         highlight_textprops=[{"color": c, "weight": "bold"} for c in colors], size=12)

# --- 4. DIRECT SAVE ---
plt.tight_layout(rect=[0, 0.03, 1, 0.9])
plt.savefig(os.path.join(FIGURES_DIR, "overall_status_landscape.png"), bbox_inches="tight")
plt.show()

## Demonstration Data

Due to the size of the numerous datasets involved, a demonstration set of data has been prepared to illustrate how the SVEF dataset was created.  src/data/create_micro_dataset.py script was used to create the demo (micro) datasets.  The (name)raw_demo datasets are intended to contain a mix of data from the original AACT data that represents nearly all aspects of the production pipeline functions.

### Load Demonstration Data

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Define the loading function with the Production Guardrail
def load_raw_demo(name):
    """
    Ingests demo data using the same Type-Hardening logic 
    as the production 'load_filtered' function.
    """
    path = os.path.join(DATA_DIR, f"{name}_micro.csv")
    df = pd.read_csv(path)
    
    # THE PRODUCTION MIRROR: Force IDs to strings immediately
    # This prevents the 'str vs int64' ValueError in Section 5
    hardened_cols = ['nct_id', 'id', 'design_group_id', 'intervention_id', 'pmid']
    for col in hardened_cols:
        if col in df.columns:
            # fillna('') ensures NaNs don't force the column back to float64/int64
            df[col] = df[col].fillna('').astype(str)
            
    return df

# 2. Re-load all 9 tables to sync them with the Production Pipeline
studies_raw = load_raw_demo('studies')
interventions_raw = load_raw_demo('interventions')
design_groups_raw = load_raw_demo('design_groups')
dg_inter_raw = load_raw_demo('design_group_interventions')
id_info_raw = load_raw_demo('id_information')
refs_raw = load_raw_demo('study_references')
sponsors_raw = load_raw_demo('sponsors')
calc_vals_raw = load_raw_demo('calculated_values')
conditions_raw = load_raw_demo('browse_conditions')

print("✅ Data Ingestion Complete: All tables synchronized with Production string-types.")

### Demonstration of Pipeline

The demonstration of the production pipeline calls specfic functions from the production scripts to demonstrate how the production pipeline works. Select visualizations are provided as well.

#### Initial Filtering and Joining

Perform initial structural filtering for Phase 2, 3, and 2/3 trials, are drug interventions, and have outcome statuses of Terminated, Suspended, Withdraw, and Unknown.  Exclude, through use of regular expression, some biologics that may be categorized as drug (and not biologic) interventions.

In [ ]:
from src.data.make_dataset import filter_structural

structural_pool, _ = filter_structural(studies_raw, interventions_raw)
print(f"Rows passing structural gate: {len(structural_pool)}")
display(structural_pool[['nct_id', 'name', 'phase', 'overall_status', 'why_stopped']].head(15))

In [ ]:
structural_pool.columns

In [ ]:
studies_raw.columns

In [ ]:
interventions_raw.columns

#### Natural Language Processinig on why_stopped

The why_stopped feature of the studies.txt file provides a description of why the trial was stopped.  Regular expression techniques are used to categorize the description.

Keyword searches are performed using Regular Expression (Regex) for three categories:
- Efficacy (futility', 'efficacy', 'lack of effect', 'benefit', 'endpoint', 'superiority', 'insufficient signal)
- Safety ('toxic', 'adverse event', 'side effect', 'harm', 'risk', 'death', 'mortality', 'aes', 'safety)
- Logistical (recruitment', 'accrual', 'enrollment', 'funding', 'covid', 'personnel', 'feasibility', 'operational)

Negation phrases are included to help reduce miscategorization (e.g. no safety concerns', 'no safety issues', 'not due to safety')

"audit_status" are assigned.  
- TERMINATED_SAFETY_CONCERN
- TERMINATED_EFFICACY_FAILURE
- TERMINATED_CLEAN_EXIT
- SUSPENDED_SAFETY_CONCERN
- SUSPENDED_EFFICACY_FAILURE
- SUSPENDED_CLEAN_EXIT
- WITHDRAWN_STRATEGIC
- WITHDRAWN_LOGISTICAL
- WITHDRAWN_OTHER

In [ ]:
from src.data.make_dataset import apply_unified_svef_logic

candidates, audit_trace = apply_unified_svef_logic(structural_pool)
display(audit_trace[['nct_id', 'audit_status', 'why_stopped']].head(10))

In [ ]:
audit_trace.columns

In [ ]:
audit_trace.head(10)

In [ ]:
%pip install highlight_text

### Plot Trial overall_status by Phase of Demo Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from highlight_text import fig_text

# 1. Filter for specific phases
phases_to_plot = ['PHASE2', 'PHASE2/PHASE3', 'PHASE3']
plot_data = audit_trace[audit_trace['phase'].isin(phases_to_plot)].copy()

# 2. Pivot the data: Rows = Phase, Columns = Overall Status
pivot_df = plot_data.groupby(['phase', 'overall_status']).size().unstack(fill_value=0)

# Ensure phases are in order and only columns that exist are used
pivot_df = pivot_df.reindex(phases_to_plot)
status_order = ['TERMINATED', 'SUSPENDED', 'WITHDRAWN', 'UNKNOWN']
existing_statuses = [c for c in status_order if c in pivot_df.columns]
pivot_df = pivot_df[existing_statuses]

# 3. Setup Plotting Variables
totals = pivot_df.sum(axis=1)
fig, ax = plt.subplots(figsize=(11, 6), dpi=100)
bar_width = 0.18 
x = np.arange(len(pivot_df.index))

# Define the Master Color Map
full_color_map = {
    'TERMINATED': "#E85252", 
    'SUSPENDED': "#E0AC2B", 
    'WITHDRAWN': "#6689C6", 
    'UNKNOWN': "#555555"
}
# Extract only colors for statuses present in this specific data slice
active_colors = [full_color_map[col] for col in pivot_df.columns]

# 4. Draw Background Totals (The Context Rectangles)
ax.bar(x, totals, width=bar_width * 5.5, color="#F2F2F2", zorder=0)

# 5. Draw the Grouped Bars
for i, status in enumerate(pivot_df.columns):
    ax.bar(
        x + (i - (len(pivot_df.columns)-1)/2) * bar_width, 
        pivot_df[status],
        width=bar_width,
        zorder=1,
        color=active_colors[i],
        label=status
    )

# 6. Styling & Formatting
ax.set_xticks(x)
ax.set_xticklabels(['Phase 2', 'Phase 2/3', 'Phase 3'], fontsize=10, fontweight='bold')
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.tick_params(axis="x", size=0, pad=10)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.set_ylabel("Number of Trials")
ax.set_xlabel("Phase")

# 7. Header and Legend 
fig.text(x=0.12, y=0.96, s="Trial overall_status by Phase", size=16, weight="bold")

# Dynamically wrap each status in brackets for the color-matching
status_labels = [f"<{col}>" for col in pivot_df.columns]
legend_string = "Tracking " + ", ".join(status_labels)

fig_text(
    x=0.12,
    y=0.91,
    s=legend_string,
    highlight_textprops=[{"color": c, "weight": "bold"} for c in active_colors],
    size=12
)

plt.tight_layout(rect=[0, 0.03, 1, 0.9])
save_path = os.path.join(FIGURES_DIR, "demo_overall_status_landscape.png")
plt.savefig(save_path, bbox_inches="tight")

print(f"✅ Plot successfully saved to: {save_path}")
plt.show()

The above plot, Trial overall_status by Phase, (using Demo Data) is meant to illustrate the distribution of phase and overall_status of the demo data. 

### Mapping Candidate Relationship within Trial

nct_id, design_group_id, and intervention_id are mapped together.  group_type is collapsed into single entry using "|" to help 

In [ ]:
from src.features.enrich_dataset import map_intervention_roles

# Production Logic: Mapping intervention roles via design groups
total_evidence_df = map_intervention_roles(audit_trace, design_groups_raw, dg_inter_raw)

print(f"Total rows in Evidence DataFrame: {len(total_evidence_df)}")
display(total_evidence_df[['nct_id', 'name', 'group_type', 'audit_status', 'intervention_id']].head(10))

In [ ]:
total_evidence_df.columns

In [ ]:
total_evidence_df['intervention_id'].value_counts()

In [ ]:
total_evidence_df['nct_id'].value_counts()

Investigation of conditions of clinical trial that results in differing number of nct_id is planned. For example, if the number of 'intervention_ids' varies per trial (representing each treatment in a trial arm), why do some nct_ids only have 1 'intervention_ids' per trial. (e.g. no placebo or comparator being ran side-by-side?) 

In [ ]:
design_groups_raw.columns

In [ ]:
design_groups_raw['group_type'].value_counts()

In [ ]:
dg_inter_raw.columns

### Feature Enrichment - Enrollment, Clinical Metadata 

Enrollment data (for whole trial), start dates, end dates, trial duration days, log transformation of enrollment, normalization of log enrollment, and normalization of trial duration days is added.  "safety score" concept is introduction by comparing normalized trial duration days and normalized log enrollment.  Safety_score is currently in early stage development.  The math and trial conditions currently bias the result (ovreall enrollment and not individual arm enrollment is used with varying number of arms per trial, Duration days are probably very bias based on design and reporting, trial duration will vary based on phase and if first in human or not.  Trial duration within phase are not idendentical).  Additionally, the method to which to judge whether a score is good (i.e. how high does a score need to be to be considered good) is not understood yet.  Consideration for comparison to successful trials in same phase, as well as phase 3 and/or phase 1 data will be performed.

Additional clinical metadata is merged and further regex based filtering for biologics is performed.

NOTE: Currently having difficulty getting LaTex to display properly with local IDE and COLAB notebook from repo.  Correcting one breaks the other.

$$ \text{Safety\_Score} = \frac{\text{Norm}(\ln(1 + E)) + \text{Norm}(D)}{2} $$

**Where:**
* $E$ is the total trial **Enrollment**.
* $D$ is the **Trial Duration** in days ($\text{Primary Completion Date} - \text{Start Date}$).
* $\text{Norm}(x) = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ (Min-Max Normalization).

In [ ]:
from src.features.enrich_dataset import feature_engineering_advanced, merge_clinical_metadata

enriched_df = merge_clinical_metadata(total_evidence_df, studies_raw, calc_vals_raw, sponsors_raw, conditions_raw)
stats_df = feature_engineering_advanced(enriched_df)

display(stats_df[['nct_id', 'enrollment', 'log_enrollment', 'Safety_Score']].head())

In [ ]:
enriched_df.columns

### Feature Enrichment Publications

Publications relating to trial and drug candidate are querried at pubmed.  DOI is extracted from study_references.txt.  Attempts to categorize references as either results based (results from trial) or background based (anything not results based, i.e. pre-clinical studies, reviews, related research).  Attempts to create a score for evidence confidence.

#### Confidence_Score

The confidence_score is an in development metric that attempts to assign increased value to SVEF candidates that have relevant pubmed publications regarding them.  Publications with results from the clinical trial are valued at 5x the weight of other types of publications (pre-clinical, all other)

$$ \text{Evidence\_Confidence} = (N_{\text{results}} \times 1.0) + (N_{\text{background}} \times 0.2) + \begin{cases} 0.5 & \text{if } N_{\text{results}} > 0 \\ 0 & \text{otherwise} \end{cases} $$

NOTE: Currently having difficulty getting LaTex to display properly with local IDE and COLAB notebook from repo.  Correcting one breaks the other.

In [ ]:
from src.features.enrich_dataset import process_publications

evidence_df = process_publications(refs_raw)
display(evidence_df.head(10))

### Feature Enrichment - PubChem

PubChem is querried with the candidate drug name to return the SMILES, Molecular Weight, and LopP (octanol-water partition coefficient).  Currently, only the first indexed valued in the returned json file is retained.  This will be reviewed and improved to retain more.  Failed attempts to obtain PubChem data are also categorized.

#### Drug Name Cleaning
Drug names are cleaned prior to querrying PubChem with a worker function during the enrich_with_pubchem_architect()

In [ ]:
# Worker function that cleans drug names for querry
from src.features.enrich_dataset import clean_drug_name

test_cases = ["Dasatinib 100 MG [Sprycel]", "Timolol 0.5% Gel", "Amlodipine Besylate"]
for drug in test_cases:
    print(f"Raw: {drug:<30} -> Cleaned: {clean_drug_name(drug)}")

#### Failure Categorization

Failures to retrieve smiles data from PubChem are categorized as PLACEBO_EQUIVALENT, LOGISTICAL_GENERIC, POSSIBLE_INTERNAL_PROPRIETARY

In [ ]:
from src.features.enrich_dataset import classify_failure

print(f"Categorizing 'AZD-1234': {classify_failure('AZD-1234')}")
print(f"Categorizing 'Placebo': {classify_failure('Placebo')}")

#### Enriching with PubChem in Action

Current biggest bottleneck in production pipeline is querrying PubChem.  PubChem has a 5 API per second limit.  Production pipeline, without cache, will seek to make >20,000 API calls.  (Could be much higher number of API calls, current counter for tracking how many API calls need to be made is buggy.)  

In [ ]:
from src.features.enrich_dataset import enrich_with_pubchem_architect

# We define a local demo cache to avoid overwriting production cache
demo_cache_path = os.path.join(DATA_DIR, 'smiles_cache_demo.csv')

print("Starting Bio-Architectural Chemical Enrichment...")
# This calls the production API-tier logic
final_enriched_demo = enrich_with_pubchem_architect(stats_df, cache_path=demo_cache_path)

# Display the results of the recovery
print("\n--- Micro-Dataset Recovery Results ---")
display(final_enriched_demo[['name', 'pubchem_cid', 'smiles', 'matched_by', 'failure_reason']].head(15))

# Validation: Count how many are now 'DTI-Ready' (have SMILES)
dti_ready_count = final_enriched_demo['is_dti_ready'].sum()
print(f"\nTotal Assets ready for Drug-Target Interaction (DTI) modeling: {dti_ready_count} / {len(final_enriched_demo)}")

## SVEF_Enriched

The SVEF_Enriched dataset (processed, no raw or interim data) is provided at data/processed/SVEF_Enriched_Final.csv.

### Visualizations of SVEF_Enriched

#### Presence of SMILES data for SVEF candidates

The SVEF_Enriched dataset has SMILES data for around 43% of candidates currently.  The Plot SMILES Data Availability below visualizes this comparison.  

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Load the Enriched Dataset
file_path = os.path.join(PROCESSED_DIR, 'SVEF_Enriched_Final.csv')

try:
    df = pd.read_csv(file_path)
    
    # 2. Check for SMILES data
    # We check for 'smiles' or 'canonical_smiles' columns common in PubChem enrichment
    smiles_col = 'smiles' if 'smiles' in df.columns else 'canonical_smiles'
    
    if smiles_col not in df.columns:
        print(f"⚠️ Warning: No SMILES column found. Available columns: {df.columns.tolist()}")
    else:
        # Calculate counts
        has_smiles = df[smiles_col].notna().sum()
        no_smiles = df[smiles_col].isna().sum()
        
        # 3. Create Visualization Data
        plot_df = pd.DataFrame({
            'Category': ['SMILES Available', 'SMILES Missing'],
            'Count': [has_smiles, no_smiles]
        })

        # 4. Plotting
        plt.figure(figsize=(8, 6), dpi=100)
        sns.set_style("white")
        
        # Use your presentation colors: Blue (#6689C6) and Red (#E85252)
        colors = ["#6689C6", "#E85252"]
        
        bars = plt.bar(plot_df['Category'], plot_df['Count'], color=colors, width=0.6, alpha=0.9)
        
        # Add count labels on top of the bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + (max(plot_df['Count']) * 0.01),
                     f'{int(height)}', ha='center', va='bottom', fontweight='bold', size=11)

        # 5. Styling
        plt.title("SMILES Data Availability", size=16, weight="bold", pad=20, loc='left')
        plt.ylabel("Number of Candidates", size=12, weight="bold")
        sns.despine()
        
        # Add a percentage success rate for the presentation
        success_rate = (has_smiles / len(df)) * 100
        plt.figtext(0.13, 0.82, f"Enrichment Success Rate: {success_rate:.1f}%", 
                    fontsize=11, color="#555555", style='italic')

        # 6. Save the Figure
        save_path = os.path.join(FIGURES_DIR, "smiles_availability_comparison.png")
        plt.tight_layout()
        plt.savefig(save_path, bbox_inches="tight")
        
        print(f"✅ Comparison plot saved to: {save_path}")
        print(f"📊 Results: {has_smiles} candidates with SMILES, {no_smiles} missing.")
        plt.show()

except NameError:
    print("Error: 'PROCESSED_DIR' or 'FIGURES_DIR' is not defined. Please run your Setup cell first.")
except FileNotFoundError:
    print(f"Error: Could not find '{file_path}'.")

#### Molecular Weight (Daltons) of SVEF candidates

Molecular weight is an attribute considered in "Lipinski's Rule of 5".  Lipinski's Rule of 5 is a traditional guideline used in drug discovery.  While note a hard rule, traditionally, molecular weights under 500 Daltons have been considered disirable for small molecule drug bioavailability.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Load the data using your verified directory variables
file_path = os.path.join(PROCESSED_DIR, 'SVEF_Enriched_Final.csv')

try:
    df = pd.read_csv(file_path)
    
    # 2. Process Molecular Weight
    # Convert to numeric and count how many were missing or failed conversion
    original_count = len(df)
    df['mw_numeric'] = pd.to_numeric(df['molecular_weight'], errors='coerce')
    
    # Points that are NaN (either were empty or were non-numeric strings)
    missing_data = df[df['mw_numeric'].isna()]
    missing_count = len(missing_data)
    
    # Drop missing values for calculations
    clean_mw = df['mw_numeric'].dropna()
    
    # 3. Handle Outliers (> 2000)
    outliers = clean_mw[clean_mw > 2000]
    outlier_count = len(outliers)
    
    # Filter data to only show up to 2000 on the histogram
    plot_data = clean_mw[clean_mw <= 2000]

    # 4. Plotting
    plt.figure(figsize=(10, 6), dpi=100)
    sns.set_style("whitegrid", {'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.3})
    
    # Create the histogram
    n, bins, patches = plt.hist(
        plot_data, 
        bins=40, 
        color="#6689C6", 
        edgecolor='white', 
        alpha=0.8
    )

    # 5. Add Text Annotations for missing and outlier data
    # Positioned in the top right of the plot area
    stats_text = (
        f"Missing/Invalid Data: {missing_count}\n"
        f"Candidates > 2000 Da: {outlier_count}"
    )
    plt.gca().text(
        0.95, 0.90, stats_text, 
        transform=plt.gca().transAxes, 
        fontsize=11, 
        verticalalignment='top', 
        horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5, edgecolor='#cccccc')
    )

    # 6. Formatting
    plt.title("Molecular Weight Distribution of Drug Candidates", size=16, weight="bold", pad=20, loc='left')
    plt.xlabel("Molecular Weight (Daltons)", size=12, weight="bold")
    plt.ylabel("Frequency", size=12, weight="bold")
    plt.xlim(0, 2000) # Explicitly limit X-axis to 2000

    # Add a vertical line for the median weight (useful for discussion)
    median_mw = mw_data.median()
    plt.axvline(median_mw, color='#E85252', linestyle='--', lw=2, label=f'Median: {median_mw:.1f} Da')
    plt.legend(frameon=False, fontsize=10)
    
    sns.despine()
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    # 7. Save the figure
    save_path = os.path.join(FIGURES_DIR, "molecular_weight_histogram.png")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    
    print(f"✅ Plot saved to: {save_path}")
    print(f"Metrics: {missing_count} missing, {outlier_count} outliers excluded from view.")
    plt.show()

except NameError:
    print("Error: 'PROCESSED_DIR' or 'FIGURES_DIR' is not defined. Please run your Setup cell first.")
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}")

#### Lipophilicity (LogP)

Lipophilicity is another attribute of "Lipinski's Rule of 5" that has been traditionally considered for small molecule drug bio availability.  Octanol-Water Partition Coefficient's less than 5 have been traditionally considered desirable.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Load the Enriched Dataset
file_path = os.path.join(PROCESSED_DIR, 'SVEF_Enriched_Final.csv')

try:
    df = pd.read_csv(file_path)
    
    # 2. Process LogP Data
    # Identify the column (PubChem often uses 'xlogp' or 'logp')
    logp_col = 'xlogp' if 'xlogp' in df.columns else 'logp'
    
    if logp_col not in df.columns:
        print(f"⚠️ Column 'logp' or 'xlogp' not found. Columns: {df.columns.tolist()}")
    else:
        # Convert to numeric and identify missing/invalid entries
        df['logp_numeric'] = pd.to_numeric(df[logp_col], errors='coerce')
        
        missing_count = df['logp_numeric'].isna().sum()
        clean_logp = df['logp_numeric'].dropna()

        # 3. Plotting
        plt.figure(figsize=(10, 6), dpi=100)
        sns.set_style("white")
        
        # Create Histogram with Density Curve
        sns.histplot(clean_logp, bins=35, color="#6689C6", kde=True, 
                     edgecolor='white', alpha=0.7, line_kws={'lw': 2.5})

        # 4. Add Statistics Annotation
        stats_text = (
            f"Total Candidates: {len(df)}\n"
            f"Valid LogP Values: {len(clean_logp)}\n"
            f"Missing/NaN: {missing_count}"
        )
        plt.gca().text(0.95, 0.90, stats_text, transform=plt.gca().transAxes, 
                       fontsize=10, verticalalignment='top', horizontalalignment='right',
                       bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.5, edgecolor='#cccccc'))

        # 5. Styling
        plt.title("Lipophilicity Profile: $\log P$ Distribution", size=16, weight="bold", pad=20, loc='left')
        plt.xlabel("Octanol-Water Partition Coefficient ($\log P$)", size=12, weight="bold")
        plt.ylabel("Frequency", size=12, weight="bold")
        
        # Add a reference line for Lipinski's Rule of 5 (LogP < 5)
        plt.axvline(5, color='#E85252', linestyle='--', alpha=0.8, label="Lipinski Limit ($\log P < 5$)")
        plt.legend(frameon=False)
        
        sns.despine()
        plt.grid(axis='y', linestyle='--', alpha=0.3)

        # 6. Save the Figure
        save_path = os.path.join(FIGURES_DIR, "logp_distribution.png")
        plt.tight_layout()
        plt.savefig(save_path, bbox_inches="tight")
        
        print(f"✅ LogP plot saved to: {save_path}")
        plt.show()

except NameError:
    print("Error: 'PROCESSED_DIR' or 'FIGURES_DIR' is not defined. Run your Setup cell first.")
except FileNotFoundError:
    print(f"Error: Could not find file at {file_path}")

### Insights and Next Steps

Perhaps not surprisingly, many of the drug candidates in these trials tend to be "compliant" with Octanol-Water Partition Coefficent and Molecular Weight for Lipinski's rule of 5.  Many features of this SVEF dataset need further review and validations.  PubChem was surprisingly robust in handling various names and returning hits on SMILES, Molecular Weight, and Octanol-Water Partition Coefficient.  Enrollment data was obtained; however, parsing individual arm enrollment data vs overall trial enrollment data is trickier than expected.  The safety score concept has potential use, after additional review and rework, to be used in combination with MW and LogP for predicting better candidates for ICAN.  

A more nuanced understanding of differences in Trial Phases is needed, as well as adding if the drug candidate is a first in human trial.  Need more EDA and background still to understand how different types of trials within the same phase can affect the trial requirements, such as enrollment, duration, and rigor of safety standards.  Drugs that have marketing approval for other indications most likely have reduced requirements compared to drugs that do not have any marketing approval yet.

Additionally, the risk of a patient in a later phase trial for their "diseased" condition may bias the safety reporting comparred to healthy volunteers in other trials. 

Four Major tasks remain:
- Review, debug, create tests, verify data in SVEF dataset.  Improve PubChem capture for candidates.
- Create a Machine Learning concept to predict which candidates are more ideal for Drug-Target-Interaction (DTI)
- Create target Protein dataset
- Train and run ICAN model


#### ICAN workflow

The graphic below is the architecure of ICAN 
Source: Kurata, Hiroyuki, and Sho Tsukiyama. “ICAN: Interpretable Cross-Attention Network for Identifying Drug and Target Protein Interactions.” PLOS ONE 17, no. 10 (2022): e0276609. https://doi.org/10.1371/journal.pone.0276609.

The purpose of the SVEF dataset is to provide the 'drug' input, as a SMILES token, into the ICAN model.

![ICAN Flow](../Background_Documents/ICAN_Flow.png)

### AACT Flat File .txt schema

The below image gives a bird's eye view of the scale of the data available; however, the data must be wrangled from numerous, but overall well defined sources.

Source: https://aact.ctti-clinicaltrials.org/schema

![AACT Schema](../docs/references/aact_schema.png)